In [1]:
import yt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import bisect
from scipy.spatial import cKDTree


In [2]:
# given plt file directory
# get the flame front point 
def read_get_flamex(d1):
    import yt
    import numpy as np
    import pandas as pd
    #load ds
    ds = yt.load(d1)
    ds.force_periodicity()
    # fields we will load per grid
    grad_keys = [
        ("boxlib", "Y(H2)_gradient_x"),
        ("boxlib", "Y(H2)_gradient_y"),
        ("boxlib", "Y(H2)_gradient_z"),
    ]
    Tkey = ("boxlib", "temp")
    
    # Collect chunks here
    chunks = []
    for g in ds.index.grids:
   
        if float(g.LeftEdge[0]) >= 0.07:
            continue
    
        # Pull minimal fields first (to build mask cheaply)
        x = g[("index", "x")].ndarray_view().ravel()
        T = g[Tkey].ndarray_view().ravel()
    
        mask = (T > 310.0) & (x < 0.07)
        if not np.any(mask):
            continue
    
        # Now pull only what you need (still per-grid, so small)
        y  = g[("index", "y")].ndarray_view().ravel()
        z  = g[("index", "z")].ndarray_view().ravel()
    
        data = {
            "x": x[mask],
            "y": y[mask],
            "z": z[mask],
            "T": T[mask],
        }
        chunks.append(pd.DataFrame(data))
    
    df_350 = pd.concat(chunks, ignore_index=True)
    #print(df_350['x'].min(),df_350['x'].max())
    x_cold_region = df_350['x'].min()-0.003

    return x_cold_region





# get flame region cut df
def get_flame_cut(d1,Tcut:float=315,x_cold_region:float=0):
    import yt
    import numpy as np
    import pandas as pd
    #load ds
    ds = yt.load(d1)
    ds.force_periodicity()
    # add the 
    ds.add_gradient_fields(("boxlib", "Y(H2)"))
    Tkey = ("boxlib", "temp")
    mass_fields = [f for f in ds.field_list if "Y(" in f[1]]
    # fields we will load per grid
    grad_keys = [
        ("boxlib", "Y(H2)_gradient_x"),
        ("boxlib", "Y(H2)_gradient_y"),
        ("boxlib", "Y(H2)_gradient_z"),
    ]
  

    # Collect chunks here
    chunks = []
    
    
    for g in ds.index.grids:
        # Quick reject: if this grid is entirely to the right of x=0.07, skip it
    
        # Pull minimal fields first (to build mask cheaply)
        x = g[("index", "x")].ndarray_view().ravel()
        T = g[Tkey].ndarray_view().ravel()
    
    
        """
        Just cut the region at the boundary of isothermal wall x=0.07
        If need to find points of flame at isothermal wall, make sure set the value larger than 0.07 (e.g. x< 0.075)
        """
        mask = (x>=x_cold_region) & (x < 0.075)
        if not np.any(mask):
            continue
    
        # Now pull only what you need (still per-grid, so small)
        y  = g[("index", "y")].ndarray_view().ravel()
        z  = g[("index", "z")].ndarray_view().ravel()
        dx = g[("index", "dx")].ndarray_view().ravel()
    
        gh2x = g[grad_keys[0]].ndarray_view().ravel()
        gh2y = g[grad_keys[1]].ndarray_view().ravel()
        gh2z = g[grad_keys[2]].ndarray_view().ravel()
        gh2mag = np.sqrt(gh2x*gh2x + gh2y*gh2y + gh2z*gh2z)
    
        data = {
            "x": x[mask],
            "y": y[mask],
            "z": z[mask],
            "T": T[mask],
            "gridsize": dx[mask],
            "gh2x": gh2x[mask],
            "gh2y": gh2y[mask],
            "gh2z": gh2z[mask],
            "gh2mag": gh2mag[mask],
        }
            # mass fractions (only masked cells)
        for f in mass_fields:
            data[f[1]] = g[f].ndarray_view().ravel()[mask]
    
        chunks.append(pd.DataFrame(data))




    if len(chunks) == 0:
        raise RuntimeError("No cells found for (T>350) & (x<0.07).")

    df_cut = pd.concat(chunks, ignore_index=True)
    df_cut['global_index'] =df_cut.index
    
    min_idx = df_cut[df_cut['T']>Tcut]['x'].idxmin()
    #print(min_idx)
    #return df_cut, [df_cut['x'].iloc[min_idx], df_cut['y'].iloc[min_idx], df_cut['z'].iloc[min_idx] ]
    return df_cut, min_idx


In [5]:
def find_pathline(df, x0,y0,z0,cutoff_value,max_length,curr_dx):
    pathline = []
    pathline_dict =[]
    find_next_point_cubic(df,x0,y0,z0,cutoff_value,max_length,curr_dx,pathline,pathline_dict)
    
    return pathline,pathline_dict




def find_next_point_cubic(df,x0,y0,z0,cutoff_value,max_length,curr_dx,pathline,pathline_dict):
    # if the line is long enough, then return
    if len(pathline)>=2:
        pt0 = pathline[0]
        ptlast = pathline[-1]
        distance = ((pt0[0] - ptlast[0] )**2  + (pt0[1] - ptlast[1]  )**2 + (pt0[2] - ptlast[2]  )**2 )**0.5
        if distance >= cutoff_value:
            return 
    if len(pathline_dict) >= max_length:
            return 

    temp_weight_dict = get_weight_w_point(df,x0,y0,z0,curr_dx)
    #print(temp_weight_dict)

    # make sure the indexes of pathline and pathline dict are aligned
    pathline.append([x0,y0,z0])
    pathline_dict.append(temp_weight_dict)


    # that's the c_i/ |c| part 
    x_dirc, y_dirc, z_dirc = 0,0,0
    next_dx  =  0
    for k,v in temp_weight_dict.items():
        x_dirc += (df['gh2x'].iloc[k] * v)
        y_dirc += (df['gh2y'].iloc[k] * v)
        z_dirc += (df['gh2z'].iloc[k] * v)
        #next_dx = max(next_dx, df['gridsize'].iloc[k])
    next_dx = curr_dx
    norm_dist = (x_dirc**2+ y_dirc**2+z_dirc**2)**0.5
    x_next = x0  - x_dirc/norm_dist * next_dx
    y_next = y0  - y_dirc/norm_dist * next_dx
    z_next = z0  - z_dirc/norm_dist * next_dx
    if z_next <=0:
        return
    #print(f'The next point is x: {x_next}, y: {y_next} , z: {z_next}')

    find_next_point_cubic(df,x_next,y_next,z_next,cutoff_value,max_length,next_dx,pathline,pathline_dict)
        
    return 




"""
need to have 2 separate method to extract points

"""

# supposed to return a directionary/list with 8 pts (index)

"""
def find_pts(df,x0,y0,z0,curr_dx):
    pts_good = []
    coord_dict = {}
    
        
    xbl, xbr = x0 - 4*curr_dx, x0 + 4*curr_dx
    ybl, ybr = y0 - 4*curr_dx, y0 + 4*curr_dx
    zbl, zbr = z0 - 4*curr_dx, z0 + 4*curr_dx

    
     
    if (xbl>0.07) or (xbr>0.07):
        xbl -= 2*curr_dx
        xbr += 2*curr_dx
        ybl -= 2*curr_dx
        ybr += 2*curr_dx
        zbl -= 2*curr_dx
        zbr += 2*curr_dx
    
    # the 01 indiation of cubic
    coord_idx = [[-1,-1,-1], [-1,-1,1], [-1,1,-1],[-1,1,1],
                [1,-1,-1], [1,-1,1], [1,1,-1],[1,1,1], ]

    print(f'for {x0}, {y0}, {z0}.')
    for ec in coord_idx:
        x_edge = xbr if ec[0] ==1 else xbl
        y_edge = ybr if ec[1] ==1 else ybl
        z_edge = zbr if ec[2] ==1 else zbl
        print(f'xedge is {x_edge}')

        subxl, subxr = min(x_edge,x0), max(x_edge,x0)
        subyl, subyr = min(y_edge,y0), max(y_edge,y0)
        subzl, subzr = min(z_edge,z0), max(z_edge,z0)

        print(f'Find from x : {subxl} to {subxr}.')
        print(f'Find from y : {subyl} to {subyr}.')
        print(f'Find from z : {subzl} to {subzr}.')

        temp_cut = df[ (df['x'] > subxl) &  (df['x'] <= subxr) & (df['y'] > subyl) &  (df['y'] <= subyr) & (df['z'] > subzl) &  (df['z'] <= subzr)]
        df_cut = temp_cut.copy()

        points = df_cut[['x','y','z']].to_numpy()
        print(points)
        tree = cKDTree(points)
        
        query_point = np.array([x0, y0, z0])
        dist, i_local = tree.query(query_point)
        print(i_local)
        row = df.iloc[int(i_local)] 
        row_cut = df.iloc[int(i_local)]

        # row inside df_cut (guaranteed in-range)
        idx_global = df.index[int(i_local)]  # original df index label
        print(f"find the near point of {df[['x','y','z']].iloc[idx_global]}.  ")
        pts_good.append(idx_global)
        coord_dict[tuple(ec)] = idx_global
        
    

    return pts_good, coord_dict
"""
def find_pts(df,x0,y0,z0,curr_dx):
    pts_good = []
    coord_dict = {}
    
    
    xbl, xbr = x0 - 4*curr_dx, x0 + 4*curr_dx
    ybl, ybr = y0 - 4*curr_dx, y0 + 4*curr_dx
    zbl, zbr = z0 - 4*curr_dx, z0 + 4*curr_dx
    #print(f' trying to find points for {x0},{y0},{z0}')
    # the 01 indiation of cubic
    coord_idx = [[-1,-1,-1], [-1,-1,1], [-1,1,-1],[-1,1,1],
                [1,-1,-1], [1,-1,1], [1,1,-1],[1,1,1], ]


    for ec in coord_idx:
        x_edge = xbr if ec[0] ==1 else xbl
        y_edge = ybr if ec[1] ==1 else ybl
        z_edge = zbr if ec[2] ==1 else zbl

        subxl, subxr = min(x_edge,x0), max(x_edge,x0)
        subyl, subyr = min(y_edge,y0), max(y_edge,y0)
        subzl, subzr = min(z_edge,z0), max(z_edge,z0)


        if subzl <0:
            subzl += curr_dx/2
            subzr += curr_dx/2

        if (subxl >0.07 ) or (subxr >0.07):
            subxl += 3*curr_dx
            subxr += 3*curr_dx

        
        temp_cut = df[ (df['x'] > subxl) &  (df['x'] <= subxr) & (df['y'] > subyl) &  (df['y'] <= subyr) & (df['z'] > subzl) &  (df['z'] <= subzr)]


        
        df_cut = temp_cut.copy()
        #
        if (df_cut.shape[0]==0):
            print(f'no point can be found for x:{subxl }:{subxr }, y: {subyl }:{subyr }, z:{subzl }:{subzr }')
        points = df_cut[['x','y','z']].to_numpy()
        tree = cKDTree(points)
        
        #print(f'subxl: {subxl}, subxr: {subxr}')
        #print(f'subyl: {subyl}, subyr: {subyr}')
        #print(f'subzl: {subzl}, subzr: {subzr}')
        #print(df_cut['x'].min(), df_cut['x'].max())
        query_point = np.array([x0, y0, z0])
        # index is the index of the original df
        # and this index need to be stored and used
        #dist, idx = tree.query(query_point)
        #print(f"for {ec} is nearest index is {idx} with distance {dist} x:{df_out['x'].loc[idx]}, y:{df_out['y'].loc[idx]}, z:{df_out['z'].loc[idx]}.")
        dist, i_local = tree.query(query_point)
        if(subxl>subxr):
            print('something wrong on x !')
            break

        if(subyl>subyr):
            print('something wrong on y !')
            break

        if(subzl>subzr):
            print('something wrong on z!')
            break
        
        idx_global = df_cut['global_index'].iloc[int(i_local)]
        
        #print(f'local index is {i_local} and global index is {idx_global}.')
        #print(f"xyz: {df[['x','y','z']].iloc[idx_global]}")
        
        pts_good.append(idx_global)
        coord_dict[tuple(ec)] = idx_global
        


    return pts_good, coord_dict








In [21]:
#print(pts1[0],pts1[1],pts1[2])
#pathline,pathdict = find_pathline(df1,pts1[0],pts1[1],pts1[2],0.0035,200,34.1e-6  )

yt : [INFO     ] 2026-02-27 19:35:19,820 Parameters: current_time              = 0.1696800000000185
yt : [INFO     ] 2026-02-27 19:35:19,821 Parameters: domain_dimensions         = [384  64  64]
yt : [INFO     ] 2026-02-27 19:35:19,821 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-02-27 19:35:19,822 Parameters: domain_right_edge         = [0.105  0.0175 0.0175]


0.06950439453124999 0.06998291015625
0.06650439453124998


In [6]:

def get_weight_w_point(df0,x0,y0,z0,curr_dx):

    df =df0.copy()
    temp_ls, _ = find_pts(df,x0,y0,z0,curr_dx)
    
    temp_dist = []
    subdf = df[['x', 'y', 'z']].loc[temp_ls]
    #print(f'the length of subdf is {subdf.shape[0]}')
    minv = 1e-12
    currmin = 100
    for i in range(subdf.shape[0]):
        tx = subdf['x'].iloc[i]
        ty = subdf['y'].iloc[i]
        tz = subdf['z'].iloc[i]
        minxyz_pre = [(tx - x0  )**2, (ty - y0  )**2 , (tz- z0  )**2 ] 
        
        minxyz = [ x for x in minxyz_pre if x>0]
        minxyz.append(currmin)
        currmin = min(minxyz)
        #print(f'for current {tx}, {ty}, {tz}, the diff is {minxyz}')

    minv = min(1e-4, currmin/1000)

    

    #print(minv)
    
    for i in range(subdf.shape[0]):
        x_diff = subdf['x'].iloc[i] - x0 if (subdf['x'].iloc[i] - x0)!=0 else minv
        y_diff = subdf['y'].iloc[i] - y0 if (subdf['y'].iloc[i] - y0)!=0 else minv
        z_diff = subdf['z'].iloc[i] - z0 if (subdf['z'].iloc[i] - z0)!=0 else minv

        
        temp_dist.append( 1/( (x_diff)**2 +   (y_diff)**2 +( z_diff)**2) )

    sum_weight = sum(temp_dist)
    weight_dict = {}
    check_ttl=0

    # find way to 
    for i in range(subdf.shape[0]):
        if subdf.index[i] not in weight_dict:
            weight_dict[subdf.index[i]] = temp_dist[i] /  sum_weight
        else:
            weight_dict[subdf.index[i]] += temp_dist[i] /  sum_weight

    #print(check_ttl)
    return weight_dict


def get_T_Mass_from_list(df,pathline,pathline_dict):
    index_col = ['x', 'y', 'z']
    header_col = [ec for ec in df.columns if 'Y('  in ec ]
    header_col.append('T')
    header_col.extend( ['gh2x','gh2y', 'gh2z'])
    full_col = index_col+ header_col
    #final_df = pd.DataFrame(columns=full_col)
    # this list combined 2 items, [index, value]
    # then sort by index. 
    final_list = []
    for i in range(len(pathline_dict)):
        temp_dict = pathline_dict[i]
        temp_target_coord = pathline[i]
        
        temp_intp_list = []
        temp_intp_list.extend(pathline[i])
        temp_intp_weight = []
        #print(temp_dict)
        w=pd.Series(temp_dict, name="weight")
        temo_subdf = df[header_col].loc[w.index].copy()
        temo_subdf["weight"] = w
        #print(temo_subdf)
        #num_cols = df_selected.select_dtypes(include="number").columns
        df_weighted = temo_subdf[header_col].multiply(temo_subdf["weight"], axis=0)
        
        for i in range(len(header_col)):
            #print(df_weighted[header_col[i]])
            temp_intp_list.append(sum(df_weighted[header_col[i]]))
        final_list.append(temp_intp_list)
    
        
    return pd.DataFrame(final_list,columns = full_col)
            
            


In [7]:

#pt1 = read_get_flamex("/Users/potato/Downloads/plt59250",315)
#df1,min_idx = get_flame_cut("/Users/potato/Downloads/plt59250",315,pt1)



,x,y,z,T,gridsize,gh2x,gh2y,gh2z,gh2mag,Y(H),Y(H2),Y(H2O),Y(H2O2),Y(HO2),Y(N2),Y(O),Y(O2),Y(OH),global_index
0,0.062754,0.000137,0.000137,298.000234,0.000273,-1.853451e-11,-6.572520e-12,-1.006804e-10,1.025829e-10,-1.749652e-17,0.01304,-4.607740e-20,3.252954e-21,1.384896e-15,0.756999,2.002840e-25,0.229961,-1.462288e-22,0
1,0.062754,0.000137,0.000410,298.000214,0.000273,-1.512035e-11,-9.876544e-13,6.391687e-11,6.568841e-11,3.715790e-17,0.01304,-4.746962e-20,3.038055e-21,-8.429508e-15,0.756999,-1.288393e-24,0.229961,-9.427782e-23,1
2,0.062754,0.000137,0.000684,298.000196,0.000273,-1.512390e-11,-1.009681e-11,5.683631e-11,5.967449e-11,6.360492e-17,0.01304,-4.522897e-20,2.823656e-21,-6.276937e-16,0.756999,-1.384084e-25,0.229961,-1.057287e-22,2
3,0.062754,0.000137,0.000957,298.000142,0.000273,-1.462297e-11,-2.144418e-11,3.708323e-11,4.526422e-11,1.820824e-17,0.01304,-3.594272e-20,2.122796e-21,-1.835485e-15,0.756999,-2.368759e-25,0.229961,-1.147436e-22,3
4,0.062754,0.000137,0.001230,298.000106,0.000273,-1.655565e-12,-8.540724e-12,1.084643e-11,1.390432e-11,-2.797661e-17,0.01304,-2.958404e-20,1.654140e-21,-6.081219e-16,0.756999,-9.005388e-26,0.229961,-8.251573e-23,4


In [8]:


pt1 = read_get_flamex("/Users/potato/Downloads/plt59250")
df1,min_idx = get_flame_cut("/Users/potato/Downloads/plt59250",305,pt1)
df1.iloc[min_idx]

yt : [INFO     ] 2026-02-28 16:25:16,372 Parameters: current_time              = 0.1696800000000185
yt : [INFO     ] 2026-02-28 16:25:16,372 Parameters: domain_dimensions         = [384  64  64]
yt : [INFO     ] 2026-02-28 16:25:16,373 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-02-28 16:25:16,373 Parameters: domain_right_edge         = [0.105  0.0175 0.0175]
yt : [INFO     ] 2026-02-28 16:25:19,210 Parameters: current_time              = 0.1696800000000185
yt : [INFO     ] 2026-02-28 16:25:19,210 Parameters: domain_dimensions         = [384  64  64]
yt : [INFO     ] 2026-02-28 16:25:19,210 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-02-28 16:25:19,211 Parameters: domain_right_edge         = [0.105  0.0175 0.0175]


x               6.553955e-02
y               6.818848e-03
z               1.879883e-04
T               3.050295e+02
gridsize        3.417969e-05
gh2x           -2.799565e+00
gh2y           -4.411489e-01
gh2z            4.391896e-01
gh2mag          2.867937e+00
Y(H)            6.967230e-15
Y(H2)           1.259659e-02
Y(H2O)          8.227769e-03
Y(H2O2)         4.617052e-05
Y(HO2)          4.407511e-06
Y(N2)           7.569044e-01
Y(O)            9.697512e-11
Y(O2)           2.222207e-01
Y(OH)           1.771128e-11
global_index    2.984253e+06
Name: 2984253, dtype: float64

In [9]:


pathline,pathdict = find_pathline(df1,df1['x'].iloc[min_idx],df1['y'].iloc[min_idx],df1['z'].iloc[min_idx],0.0035,250,34.1e-6  )
df_p1 = get_T_Mass_from_list(df1,pathline,pathdict)


In [10]:
print(df_p1)
cline = []
cline.append([1- df_p1['Y(H2)'] / 0.01304])
t_profile = []
t_profile.append([df_p1['T']])
t_profile

            x         y         z          Y(H)     Y(H2)    Y(H2O)   Y(H2O2)  \
0    0.065540  0.006819  0.000188  6.967230e-15  0.012597  0.008228  0.000046   
1    0.065573  0.006824  0.000183  8.924437e-15  0.012499  0.008950  0.000052   
2    0.065606  0.006829  0.000179  1.413318e-14  0.012380  0.009726  0.000058   
3    0.065640  0.006835  0.000175  1.189869e-14  0.012236  0.010595  0.000065   
4    0.065673  0.006840  0.000173 -3.348533e-14  0.012046  0.011685  0.000075   
..        ...       ...       ...           ...       ...       ...       ...   
100  0.068865  0.007393  0.000228  9.659924e-06  0.000098  0.148359  0.000011   
101  0.068898  0.007395  0.000221  8.645411e-06  0.000097  0.148032  0.000012   
102  0.068931  0.007399  0.000229  9.146089e-06  0.000096  0.147923  0.000011   
103  0.068965  0.007402  0.000225  8.795135e-06  0.000095  0.147579  0.000011   
104  0.068999  0.007404  0.000225  8.668297e-06  0.000094  0.147370  0.000011   

       Y(HO2)     Y(N2)    

[[0       305.029510
  1       307.763234
  2       311.395636
  3       316.313086
  4       323.990408
            ...     
  100    1179.040426
  101    1149.851534
  102    1171.732271
  103    1164.173424
  104    1164.631879
  Name: T, Length: 105, dtype: float64]]

In [11]:
df_whole_front = df1[(df1['x']<=0.07) & (df1['z'] > 2*34e-6) & (df1['y']<=0.015) & (df1['y'] >=0.002) & (df1['T']>=305)]
df_sub = df_whole_front.loc[df_whole_front.groupby('y')['x'].idxmin()].reset_index(drop=True)

In [12]:
xs2 = df_sub['x'].iloc[1]
ys2 = df_sub['y'].iloc[1]
zs2 = df_sub['z'].iloc[1]
print(f'try this {xs2}, {ys2},{zs2}.')

try this 0.06967529296874998, 0.0020336914062500005,8.544921875000001e-05.


In [13]:
pl2,pd2 = find_pathline(df1,  xs2  ,  ys2 ,  zs2  ,0.0035,250,68.1e-6  )
df_p2 = get_T_Mass_from_list(df1,pl2,pd2)

In [ ]:
sf_cline= []
sf_t = []
for i in range(df_sub.shape[0]):
    temp_xs  = df_sub['x'].iloc[i]
    temp_ys  = df_sub['y'].iloc[i]
    temp_zs  = df_sub['z'].iloc[i]
    try:
        tpl2,tpd2 = find_pathline(df1,  temp_xs  ,  temp_ys ,  temp_zs  ,0.0035,250,68.1e-6  )
        temp_df2 = get_T_Mass_from_list(df1,tpl2,tpd2) 
        temp_c2 = [1- temp_df2['Y(H2)'] /0.01304 ]
        temp_t2 = [temp_df2['T'] ]
        sf_cline.append(temp_c2)
        sf_t.append(temp_t2)
        print(f'The point of {temp_xs}, {temp_ys},{temp_zs} find c line.')
    except:
        print(f'The point of {temp_xs}, {temp_ys},{temp_zs} cant get progress variable line')


The point of 0.06969238281249998, 0.0020166015625000004,0.00010253906250000001 find c line.
The point of 0.06967529296874998, 0.0020336914062500005,8.544921875000001e-05 find c line.
The point of 0.06986328124999999, 0.00205078125,0.00013671875 find c line.
The point of 0.06967529296874998, 0.00206787109375,8.544921875000001e-05 find c line.
The point of 0.06969238281249998, 0.0020849609375000002,0.00010253906250000001 find c line.
The point of 0.06967529296874998, 0.0021020507812500003,8.544921875000001e-05 find c line.
The point of 0.06979492187499999, 0.0021191406250000004,6.8359375e-05 find c line.
The point of 0.06967529296874998, 0.00213623046875,8.544921875000001e-05 find c line.
The point of 0.06969238281249998, 0.0021533203125,0.00010253906250000001 find c line.
The point of 0.06967529296874998, 0.00217041015625,8.544921875000001e-05 find c line.
The point of 0.06967529296874998, 0.0022045898437500003,8.544921875000001e-05 find c line.
The point of 0.06969238281249998, 0.00222